In [ ]:
# Find this checkout from a notebook or a kernel launched at the workspace root.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents, _here / 'cic17-cic18-transfer-study']
if os.environ.get('NIDS_STUDY_ROOT'):
    _candidates.insert(0, Path(os.environ['NIDS_STUDY_ROOT']).expanduser().resolve())
REPO_ROOT = next((p for p in _candidates if (p / 'src/nids_transfer').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this checkout or set NIDS_STUDY_ROOT.')
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
from nids_transfer.paths import study_paths
DATA_DIR, ARTIFACT_DIR = study_paths(REPO_ROOT)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_DIR)


# Cross-year flow clustering with HDBSCAN

Three complementary questions:

1. **All traffic:** do label-blind clusters naturally group attack categories and mix years?
2. **Benign only:** do normal flows from the two years occupy shared dense regions?
3. **Selected shared labels:** are comparable, identically named attacks shared across years?

This notebook reads the compatible Parquet files and never changes them. Full-data preprocessing is memory-bounded; HDBSCAN fits on an equal-size random sample from each year. Protocol/flag variants reuse exactly the same rows. Source row numbers plus year identify samples reliably, independently of potentially duplicated flow IDs.

The implementation lives in `src/nids_transfer/dataset_clustering_utils.py`. Requires the `IMT_data_analyser` environment: DuckDB, PyArrow, pandas, NumPy, matplotlib, and scikit-learn with HDBSCAN. This uses **scikit-learn HDBSCAN**; its `min_samples` includes the point itself. No approximate assignment of the remaining source rows is performed.
Defaults: 50,000 rows per year for all traffic, 20,000 per year for benign-only runs, and up to 5,000 per year for selected attack labels. Dense benign samples are smaller to keep HDBSCAN runtime practical.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from nids_transfer.dataset_clustering_utils import (
    FlowStudy, fit_auxiliary, run_experiment, plot_report, stability,
)

ROOT = REPO_ROOT
PATHS = {year: DATA_DIR / f'compatible/CIC{year}_shared.parquet'
         for year in ['2017', '2018']}
ROWS_PER_YEAR = 50_000
BENIGN_ROWS_PER_YEAR = 20_000
SEED = 42
N_PCS = 12
MIN_CLUSTER_SIZE = 500
MIN_SAMPLES = 30
PROTOCOL_WEIGHT = 0.5
FLAG_WEIGHT = 0.5
RUN_ATTACK_EXPERIMENTS = True
ATTACK_LABELS = ['DoS Hulk', 'DoS GoldenEye']
RUN_ROBUSTNESS_GRID = False  # Opt-in larger grid below; one 9-PC check runs by default.

OUT = ARTIFACT_DIR / 'clustering_results'
OUT.mkdir(exist_ok=True)
study = FlowStudy(PATHS)
results = {}


## Label inventory and cleaning

Keep original labels and attempted variants separate. The `family` column is an **exploratory coarse grouping**, used only after clustering; membership in the same broad family does not establish equivalent attacks or labeling practices. Inspect the complete exported mapping. The existing CIC18 JSON maps capture filenames, so it cannot directly harmonize these flow labels.

Exact shared raw labels are candidates for label-specific analysis, subject to their counts and semantics. Missing/NaN/infinite numeric rows receive the same exclusion as the PCA notebook. Negative continuous values become missing before log1p, with pooled log-mean imputation. Zero-traffic flows remain included.
The exported mapping is an audit table. To change coarse grouping, edit `coarse_family()` in the helper module; editing the exported CSV alone does not change the analysis.


In [ ]:
inventory = study.inventory()
display(inventory)
label_mapping = inventory[['label', 'family', 'attempted_variant']].drop_duplicates().sort_values('label')
display(label_mapping)
shared_labels = sorted(set(inventory.query("year == '2017'").label)
                       & set(inventory.query("year == '2018'").label))
print('Exact shared labels:', shared_labels)
print('Flag count columns:', study.flags)
label_mapping.to_csv(OUT / 'label_mapping_review.csv', index=False)
inventory.to_csv(OUT / 'full_label_inventory.csv', index=False)


## Shared full-data PCA and benign-specific PCA

Reproduce the comparison notebook's continuous-feature geometry: exclude metadata, protocol/ports/ICMP, and all flag fields; fit column-wise StandardScaler on log1p observations; fill missing standardized values with zero; and diagonalize the full pooled covariance. A separate benign-only fit is pooled across both years as well. Neither fit sees year or label as a feature.

The all-traffic fit is row-weighted across the complete source populations, so 2017 influences it more. The **HDBSCAN samples** below are year-balanced. These are distinct study choices: balancing clustering densities does not rebalance the previously chosen PCA basis. Benign-specific axes and cluster IDs are separate from the global analysis.

Twelve PCs are the initial global choice (~95% pooled variance); the benign-specific fit selects its own count to reach 95%. Unwhitened scores preserve relative PC variances.

In [ ]:
global_space = study.fit_space()
benign_space = study.fit_space(label='BENIGN')
BENIGN_PCS = max(2, int(np.searchsorted(np.cumsum(benign_space['ratio']), 0.95)) + 1)
for name, space, count in [('global', global_space, N_PCS), ('benign', benign_space, BENIGN_PCS)]:
    print(name, 'population rows:', space['counts'], 'retained PCs:', count,
          'variance retained:', round(space['ratio'][:count].sum(), 4))
    display(space['audit'].query('negative_to_missing > 0'))
    np.savez_compressed(OUT / f'{name}_preprocessing.npz',
        feature_order=np.array(study.features), active=space['active'],
        log_mean=space['scaler'].mean_, log_scale=space['scaler'].scale_,
        log_variance=space['scaler'].var_, mean=space['mean'],
        components=space['components'], eigenvalues=space['eigenvalues'])


In [ ]:
all_sample = study.sample(ROWS_PER_YEAR, SEED)
benign_sample = study.sample(BENIGN_ROWS_PER_YEAR, SEED, label='BENIGN')
all_aux = fit_auxiliary(all_sample, study)
benign_aux = fit_auxiliary(benign_sample, study)
print('All-traffic sample counts:')
display(pd.crosstab(all_sample.label, all_sample.year))
print('Benign sample counts:')
display(benign_sample.groupby('year').size())
display(all_aux['flag_audit'])
sample_inventory = pd.crosstab(all_sample.label, all_sample.year)
sample_inventory.to_csv(OUT / 'all_sample_label_counts.csv')
# Small labels may be absent in the representative random sample; they are not oversampled.


## Weighted distance blocks

For continuous scores T, use `T / sqrt(2 × sum(retained eigenvalues))`. This rescales the **whole block**, retaining relative PC weights. Its expected squared distance between independent pooled population rows is approximately 1.

Protocol uses full one-hot encoding, multiplied by `sqrt(protocol_weight / 2)`: a mismatch adds exactly `protocol_weight` to squared distance. Do not z-score protocol indicators.

These datasets contain genuine flag counts (not just 0/1 indicators). Apply log1p to nonnegative counts, center using the pooled experiment sample, mean-impute negative sentinels if present, remove constants, then scale the whole block by `sqrt(flag_weight / (2 × sum(flag variances)))`. No per-flag z-scoring amplifies rare flags. Weights are explicit modeling choices, not learned truths. Ports/ICMP codes remain excluded.

All protocols use a shared vocabulary within an experiment. PCA, auxiliary-block fitting, and sampling never use attack labels for the all-traffic run. The only label conditioning is deliberate subset selection in benign/attack-specific experiments.

In [ ]:
def run_and_save(name, frame, space=global_space, aux=all_aux, **kwargs):
    params = dict(n_pcs=N_PCS, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES)
    params.update(kwargs)
    result = run_experiment(name, frame, study, space, aux, **params)
    results[name] = result
    result['assignments'].to_csv(OUT / f'{name}_assignments.csv', index=False)
    for report_name, table in result['reports'].items():
        table.to_csv(OUT / f'{name}_{report_name}.csv')
    print(name, result['parameters'])
    display(result['reports']['overview'])
    display(result['reports']['noise_by_year'])
    return result

# Start from continuous PCs, then add protocol, then flags on identical sampled rows.
all_pc = run_and_save('all_pc', all_sample)
all_protocol = run_and_save('all_pc_protocol', all_sample, protocol_weight=PROTOCOL_WEIGHT)
all_flags = run_and_save('all_pc_protocol_flags', all_sample,
                         protocol_weight=PROTOCOL_WEIGHT, flag_weight=FLAG_WEIGHT)


## Do clusters naturally recover categories?

Cluster IDs have meaning only inside one fitted model. Noise label −1 is **not** a shared cluster. Membership strength is the model's assignment strength, not a calibrated probability of an attack or of correctness.

Inspect both normalization directions: cluster composition shows label purity; label capture shows fragmentation and the portion of each label recovered. Capture denominators include noise. Homogeneity, completeness, ARI, and AMI are computed on assigned rows, accompanied by coverage and an attacks-only view. A high score after rejecting most difficult rows can mislead. Coarse-family scores complement, but do not replace, raw-label scores.

Year composition uses the sampled input baseline, not the 80/20 full-data year ratio. `conditional_year` additionally compares year shares within each raw label/family to that category's own year baseline; this distinguishes category-prevalence differences from residual cross-year separation. Inspect small cell counts before interpreting enrichment.
With a minimum cluster size of 500, rare labels cannot form their own standalone clusters. Absence from a cluster is not evidence that a rare attack is indistinguishable; consult the full and sampled inventories.


In [ ]:
display(all_pc['reports']['alignment'])
display(all_pc['reports']['noise_by_label'])
display(all_pc['reports']['year_enrichment'])
display(all_pc['reports']['conditional_year'].head(30))
plot_report(all_pc)
# Full raw-label tables (also exported as CSV):
display(all_pc['reports']['label_cluster_composition'])
display(all_pc['reports']['label_capture'])

# Raw-label heatmaps for the baseline; noise remains in capture denominators.
import matplotlib.pyplot as plt
for key, title in [('label_cluster_composition', 'Raw label composition of each cluster'),
                   ('label_capture', 'Fraction of each raw label captured by each cluster')]:
    table = all_pc['reports'][key]
    if not table.empty:
        fig, ax = plt.subplots(figsize=(17, min(20, max(5, len(table) * 0.25))), constrained_layout=True)
        image = ax.imshow(table.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='viridis')
        ax.set_xticks(range(len(table.columns)), table.columns, rotation=90, fontsize=7)
        ax.set_yticks(range(len(table)), table.index)
        ax.set_ylabel('Cluster'); ax.set_title(title)
        fig.colorbar(image, ax=ax, label='Fraction'); plt.show()


In [ ]:
benign_global = run_and_save('benign_global_pc', benign_sample, aux=benign_aux)
plot_report(benign_global)
display(benign_global['reports']['year_enrichment'])


In [ ]:
benign_local = run_and_save('benign_refit_pc', benign_sample, space=benign_space, aux=benign_aux, n_pcs=BENIGN_PCS)
plot_report(benign_local)
display(benign_local['reports']['year_enrichment'])
# Different preprocessing spaces: compare composition/stability, not raw PC coordinates.


## Optional exact-label experiments

Use identically named, non-attempted `DoS Hulk` and `DoS GoldenEye` labels as initial cases. This is a narrower comparison than a coarse attack-family match, but semantics can still differ across collections. Reuse the global PCA to keep the overall geometry. Sample at most 5,000 rows per year; when a label has fewer rows, year counts will be unequal. Enrichment automatically uses the actual sample baseline.

These small subsets use a smaller minimum cluster size (100). Results cannot be compared to all-traffic clusters merely by numeric cluster ID.

In [ ]:
if RUN_ATTACK_EXPERIMENTS:
    for label in ATTACK_LABELS:
        if label not in shared_labels:
            print('Skipping label absent from one year:', label)
            continue
        frame = study.sample(per_year=5_000, seed=SEED, label=label)
        if frame.groupby('year').size().min() < 100:
            print('Skipping insufficient rows:', label)
            continue
        name = 'label_' + label.lower().replace(' ', '_')
        subset = run_and_save(name, frame, aux=fit_auxiliary(frame, study),
                              min_cluster_size=100)
        plot_report(subset)


## Sensitivity and sampling stability

A 9-versus-12-PC check runs by default on exactly the same all-traffic rows. Representation variants above also reuse identical rows. Compare assignment agreement after matching stable row keys: ARI is evaluated only on points assigned in both runs, alongside noise-status agreement and the number of common rows. No numeric cluster IDs are matched directly.

The optional grid changes one factor at a time: cluster size, `min_samples`, block weights, or sampling seed. Between seed samples, only their overlapping original rows contribute to stability metrics; small overlap limits the comparison. Parameter values here are scikit-learn's convention. Do not tune toward a desired degree of year mixing or label purity; report sensitivity rather than selecting the prettiest outcome.

In [ ]:
all_pc9 = run_and_save('all_pc9', all_sample, n_pcs=9)
if RUN_ROBUSTNESS_GRID:
    for size in [100, 1000]:
        run_and_save(f'all_pc_size{size}', all_sample, min_cluster_size=size)
    for minimum in [15, 60]:
        run_and_save(f'all_pc_samples{minimum}', all_sample, min_samples=minimum)
    for weight in [0.25, 1.0]:
        run_and_save(f'all_protocol_weight{weight}', all_sample, protocol_weight=weight)
        run_and_save(f'all_flags_weight{weight}', all_sample,
                     protocol_weight=PROTOCOL_WEIGHT, flag_weight=weight)
    for seed in [7, 123]:
        frame = study.sample(ROWS_PER_YEAR, seed)
        run_and_save(f'all_pc_seed{seed}', frame, aux=fit_auxiliary(frame, study))

stability_table = pd.DataFrame([stability(all_pc, r) for name, r in results.items()
                                if name.startswith('all_') and name != 'all_pc'])
display(stability_table)
stability_table.to_csv(OUT / 'stability.csv', index=False)
summary_rows = []
for name, result in results.items():
    row = dict(experiment=name, **result['parameters'], **result['reports']['overview'].iloc[0].to_dict())
    summary_rows.append(row)
experiment_summary = pd.DataFrame(summary_rows).set_index('experiment')
display(experiment_summary)
experiment_summary.to_csv(OUT / 'experiment_summary.csv')

import sklearn, duckdb, platform
manifest = {
    'seed': SEED, 'rows_per_year_requested': ROWS_PER_YEAR,
    'benign_rows_per_year_requested': BENIGN_ROWS_PER_YEAR,
    'category_policy': 'Original labels retained; coarse families exploratory; attempted variants separate in raw labels',
    'pca_fit_weighting': 'Equal weight per full-population cleaned row; global and benign fits pooled across years',
    'clustering_sample_weighting': 'Equal requested rows per year; no category balancing',
    'versions': {'python': platform.python_version(), 'sklearn': sklearn.__version__,
                 'duckdb': duckdb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__},
    'inputs': {year: {'path': str(path.resolve()), 'bytes': path.stat().st_size,
                      'mtime_ns': path.stat().st_mtime_ns} for year, path in PATHS.items()},
    'experiments': {name: result['parameters'] for name, result in results.items()},
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print('Study artifacts:', OUT.resolve())


## How much of each year is in shared clusters?

Presence of one point from another year can be misleading. The table below uses a minimum of 10 points from each year and reports both the share of all sampled rows and the share of assigned rows in these clusters. Noise remains in the first denominator. Review this threshold alongside cluster size; it is not a test of distributional equality.


In [ ]:
# Require at least 10 sampled points from EACH year to call a cluster shared here.
# This is a descriptive count threshold, not a statistical equivalence test.
mixing_rows = []
for name, result in results.items():
    table = result['reports']['mixing_by_year'].reset_index()
    table.insert(0, 'experiment', name)
    mixing_rows.append(table)
year_mixing_summary = pd.concat(mixing_rows, ignore_index=True)
display(year_mixing_summary)
year_mixing_summary.to_csv(OUT / 'year_mixing_summary.csv', index=False)

# Same benign sample, different preprocessing spaces: cluster IDs need not match.
benign_stability = pd.DataFrame([stability(results['benign_global_pc'], results['benign_refit_pc'])])
display(benign_stability)
benign_stability.to_csv(OUT / 'benign_preprocessing_stability.csv', index=False)


## Interpretation and next steps

- A mixed-year cluster is evidence of a shared dense region under this representation, not proof of distributional equivalence. A cluster with only one year may reflect sample size, category prevalence, protocol, capture artifacts, or real differences.
- Noise points are analyzed individually/by label; they are not one cluster. Low cross-year mixing among a tiny number of retained points is inconclusive.
- PC1–PC2 plots omit variation used by HDBSCAN in the other PCs. Colors can overlap even when full-dimensional clusters are separated.
- Benign-global and benign-refit experiments address different geometric questions. Prefer conclusions robust to both, to PC count, and to sampling.
- Clustering is transductive here: no out-of-sample attack classifier or approximate full-data assignment is claimed. Future family-specific runs should follow review of the exported label mapping.
- Rerunning replaces study artifacts under `clustering_results/`; source datasets remain untouched. Auxiliary flag scales are fitted on each experiment's sample. Saved assignments and manifest identify exactly which rows and settings were used.

Useful objects: `results`, `experiment_summary`, `stability_table`, and each result's `assignments` / `reports`. Every report is also exported to CSV for inspection outside the notebook.